# Clusterización K-Means esférica de coordenadas

1. Carga los parquets de *train* y arma un dataset de `lat`/`lon` (una fila por locación: se descartan las repeticiones).
2. Mapa con todas las coordenadas.
3. K-Means esférico (distancia angular / similitud coseno sobre la esfera unitaria).
4. Mapa con los centroides y su diagrama de Voronoi esférico.

Dependencias: `pip install numpy pandas pyarrow scipy plotly`

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pyarrow.parquet as pq
from scipy.spatial import SphericalVoronoi

# ---- Configuración ----
# Buscar el .env desde la carpeta del notebook o la raíz del repositorio.
env_path = next(
    (folder / ".env" for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError("No se encontró el archivo .env del proyecto.")

# Cargar DATA_URL sin agregar dependencias al proyecto.
for line in env_path.read_text().splitlines():
    key, separator, value = line.partition("=")
    if separator and key.strip() == "DATA_URL":
        os.environ.setdefault("DATA_URL", value.strip().strip("\"'"))

DATA_URL = os.environ.get("DATA_URL", "")
if not DATA_URL:
    raise ValueError("Configurá DATA_URL en el archivo .env.")

data_dir = Path(DATA_URL).expanduser()
if not data_dir.is_absolute():
    data_dir = env_path.parent / data_dir
data_dir = data_dir.resolve()
if not data_dir.is_dir():
    raise NotADirectoryError(f"La carpeta de datos no existe: {data_dir}")
K = 100               # cantidad de clusters
N_INIT = 5           # reinicios de k-means (se queda con el mejor)
MAX_ITER = 100
TOL = 1e-7
SEED = 42
MAX_PLOT_POINTS = 50_000   # tope de puntos dibujados en el mapa (solo afecta al plot)

print(data_dir)

/Users/juanmanueljuradokokoyaczuk/Documents/facultad/redes-neuronales-2/tp-final/world-streetview-500k/data


## 1. Carga de parquets y dataset de coordenadas

In [2]:
LAT_NAMES = ["lat", "latitude", "latitud"]
LON_NAMES = ["lon", "long", "lng", "longitude", "longitud"]

def find_col(names_in_file, candidates):
    lower = {n.lower(): n for n in names_in_file}
    for c in candidates:
        if c in lower:
            return lower[c]
    raise KeyError(f"No encontré ninguna de {candidates} en {list(names_in_file)}")

files = sorted(data_dir.rglob("*.parquet"))
if not files:
    raise FileNotFoundError(f"No hay .parquet en {data_dir.resolve()}")
print(f"{len(files)} parquets en {data_dir.resolve()}")

parts, total_rows = [], 0
for f in files:
    schema_names = pq.ParquetFile(f).schema_arrow.names
    lat_c, lon_c = find_col(schema_names, LAT_NAMES), find_col(schema_names, LON_NAMES)
    df = pq.read_table(f, columns=[lat_c, lon_c]).to_pandas()  # solo 2 columnas: liviano
    df.columns = ["lat", "lon"]
    total_rows += len(df)
    # Deduplicar por archivo para no acumular las 4 repeticiones en memoria
    parts.append(df.round(6).drop_duplicates())

coords = pd.concat(parts, ignore_index=True).drop_duplicates().dropna().reset_index(drop=True)
coords = coords[coords.lat.between(-90, 90) & coords.lon.between(-180, 180)].reset_index(drop=True)
print(f"filas leídas: {total_rows:,} -> locaciones únicas: {len(coords):,}")
coords.head()

394 parquets en /Users/juanmanueljuradokokoyaczuk/Documents/facultad/redes-neuronales-2/tp-final/world-streetview-500k/data
filas leídas: 2,002,060 -> locaciones únicas: 502,760


,lat,lon
0,23.537520,58.184800
1,-11.430396,-61.437989
2,57.700562,12.616149
3,36.898451,-98.056079
4,45.240655,-89.039845


## 2. Mapa con todas las coordenadas

In [3]:
def base_layout(fig, title):
    fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4",
                    showocean=True, oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb",
                    showcoastlines=True, coastlinecolor="#888888")
    fig.update_layout(title=title, height=620, margin=dict(l=0, r=0, t=50, b=0), legend=dict(x=0.01, y=0.01))
    return fig

plot_df = coords if len(coords) <= MAX_PLOT_POINTS else coords.sample(MAX_PLOT_POINTS, random_state=SEED)
fig = go.Figure(go.Scattergeo(lat=plot_df.lat, lon=plot_df.lon, mode="markers",
                              marker=dict(size=3, color="#1f77b4", opacity=0.6), name="locaciones"))
base_layout(fig, f"Todas las coordenadas ({len(plot_df):,} de {len(coords):,})").show()

## 3. K-Means esférico

Cada punto se pasa a un vector unitario en $\mathbb{R}^3$. La asignación es por mayor producto punto
(menor distancia angular / geodésica) y el centroide de un cluster es la media de sus vectores
**reproyectada a la esfera** (normalizada). Inicialización tipo k-means++ usando distancia angular.

In [4]:
def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

def to_latlon(xyz):
    xyz = xyz / np.linalg.norm(xyz, axis=1, keepdims=True)
    return np.degrees(np.arcsin(np.clip(xyz[:, 2], -1, 1))), np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))

def assign(X, C, chunk=200_000):
    labels = np.empty(len(X), dtype=np.int32)
    sim = np.empty(len(X))
    for i in range(0, len(X), chunk):
        s = X[i:i + chunk] @ C.T
        labels[i:i + chunk] = s.argmax(axis=1)
        sim[i:i + chunk] = s.max(axis=1)
    return labels, sim

def init_pp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    d = 1 - np.clip(X @ C[0], -1, 1)
    for _ in range(1, k):
        C.append(X[rng.choice(len(X), p=d / d.sum())])
        d = np.minimum(d, 1 - np.clip(X @ C[-1], -1, 1))
    return np.array(C)

def spherical_kmeans(X, k, n_init=5, max_iter=100, tol=1e-7, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        C = init_pp(X, k, rng)
        prev = -np.inf
        for _ in range(max_iter):
            labels, sim = assign(X, C)
            for j in range(k):
                m = labels == j
                if m.any():
                    v = X[m].sum(axis=0)
                    C[j] = v / np.linalg.norm(v)
                else:  # cluster vacío: reubicar en el punto peor representado
                    C[j] = X[sim.argmin()]
            score = sim.sum()
            if score - prev < tol * abs(score):
                break
            prev = score
        labels, sim = assign(X, C)
        if best is None or sim.sum() > best[0]:
            best = (sim.sum(), C.copy(), labels)
    return best[1], best[2]

X = to_xyz(coords.lat.to_numpy(), coords.lon.to_numpy())
centroids, labels = spherical_kmeans(X, K, N_INIT, MAX_ITER, TOL, SEED)
coords["cluster"] = labels
c_lat, c_lon = to_latlon(centroids)
cent = pd.DataFrame({"cluster": range(K), "lat": c_lat, "lon": c_lon,
                     "n": np.bincount(labels, minlength=K)})
ang = np.degrees(np.arccos(np.clip((X * centroids[labels]).sum(axis=1), -1, 1)))
print(f"distancia angular media al centroide: {ang.mean():.3f}° (~{ang.mean() * 111.19:.0f} km)")
cent

distancia angular media al centroide: 2.459° (~273 km)


,cluster,lat,lon,n
0,0,41.233947,-89.663408,9045
1,1,-26.628208,28.475492,4759
2,2,47.671400,15.952897,10216
3,3,-21.878143,145.721046,1555
4,4,17.606145,76.410030,11790
...,...,...,...,...
95,95,56.097090,55.966469,3265
96,96,-13.720090,-72.585551,1979
97,97,41.016387,-6.944121,6350
98,98,38.353138,35.225378,5591


## 4. Centroides + diagrama de Voronoi esférico

In [5]:
def slerp_arc(a, b, n=24):
    omega = np.arccos(np.clip(a @ b, -1, 1))
    if omega < 1e-9:
        return np.array([a, b])
    t = np.linspace(0, 1, n)[:, None]
    return (np.sin((1 - t) * omega) * a + np.sin(t * omega) * b) / np.sin(omega)

def voronoi_edges(centroids):
    sv = SphericalVoronoi(centroids, radius=1.0)
    sv.sort_vertices_of_regions()
    seen, lats, lons = set(), [], []
    for region in sv.regions:
        for i in range(len(region)):
            u, v = region[i], region[(i + 1) % len(region)]
            key = (min(u, v), max(u, v))
            if key in seen:
                continue
            seen.add(key)
            la, lo = to_latlon(slerp_arc(sv.vertices[u], sv.vertices[v]))
            # cortar la línea si cruza el antimeridiano
            jump = np.where(np.abs(np.diff(lo)) > 180)[0]
            la, lo = list(la), list(lo)
            for j in reversed(jump):
                la.insert(j + 1, None); lo.insert(j + 1, None)
            lats += la + [None]; lons += lo + [None]
    return lats, lons

e_lat, e_lon = voronoi_edges(centroids)

fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=e_lat, lon=e_lon, mode="lines",
                            line=dict(width=1.5, color="#d62728"), name="Voronoi"))
fig.add_trace(go.Scattergeo(lat=cent.lat, lon=cent.lon, mode="markers+text",
                            text=cent.cluster.astype(str), textposition="top center",
                            marker=dict(size=9, color="black", symbol="x"),
                            customdata=cent.n, hovertemplate="cluster %{text}<br>n=%{customdata}<extra></extra>",
                            name="centroides"))
base_layout(fig, f"Centroides esféricos (k={K}) y diagrama de Voronoi").show()

## 5. Patrones (lat, lon) por celda de Voronoi

Cada locación (patrón) pertenece a la celda de su prototipo más cercano. La tabla tiene una fila por patrón, con el prototipo asociado y la distancia geodésica a él; abajo, un resumen por celda.

In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


# Tabla de patrones: una fila por (lat, lon) con su prototipo/celda de Voronoi
patrones = coords.rename(columns={"lat": "lat", "lon": "lon"}).copy()
patrones["proto_lat"] = cent.lat.to_numpy()[labels]
patrones["proto_lon"] = cent.lon.to_numpy()[labels]
patrones["dist_km"] = np.radians(ang) * 6371.0
patrones = (patrones[["cluster", "proto_lat", "proto_lon", "lat", "lon", "dist_km"]]
            .sort_values(["cluster", "dist_km"]).reset_index(drop=True))

# Resumen por celda
resumen = (patrones.groupby("cluster")
           .agg(proto_lat=("proto_lat", "first"), proto_lon=("proto_lon", "first"),
                n_patrones=("lat", "size"), dist_media_km=("dist_km", "mean"), dist_max_km=("dist_km", "max"),
                lat_min=("lat", "min"), lat_max=("lat", "max"), lon_min=("lon", "min"), lon_max=("lon", "max"))
           .round(3))

resumen.sort_values("n_patrones", ascending=False, inplace=True)
display(resumen)

# Para ver los patrones de una celda concreta: patrones[patrones.cluster == 0]
# Para guardar la tabla completa: patrones.to_csv("patrones_por_celda.csv", index=False)
#display(patrones)

,proto_lat,proto_lon,n_patrones,dist_media_km,dist_max_km,lat_min,lat_max,lon_min,lon_max
cluster,,,,,,,,,
42,23.397,73.300,12126,254.213,548.235,19.594,27.630,68.503,78.029
4,17.606,76.410,11790,258.098,530.467,13.762,21.297,72.764,81.148
17,44.751,1.788,11468,207.785,579.929,39.853,47.658,-3.018,5.899
60,28.159,76.598,11129,244.692,678.137,23.987,34.196,71.873,80.918
15,15.657,101.439,10806,248.368,1115.487,10.239,20.452,92.542,105.374
26,24.578,89.353,10790,263.875,783.424,20.457,28.620,85.663,96.359
2,47.671,15.953,10216,228.447,505.299,43.618,50.896,11.203,21.243
99,35.473,-86.884,10015,215.320,416.019,32.034,38.519,-91.391,-83.477
30,51.919,20.054,9882,212.258,672.545,48.391,55.582,15.030,29.670


In [7]:
from plotly.subplots import make_subplots

# Distribución de n_patrones por celda de Voronoi
fig = make_subplots(rows=1, cols=2, subplot_titles=("Histograma de n_patrones", "n_patrones por celda (ordenado)"))
fig.add_trace(go.Histogram(x=resumen.n_patrones, nbinsx=30, marker_color="#1f77b4", name="celdas"), row=1, col=1)
srt = resumen.n_patrones.sort_values(ascending=False)
fig.add_trace(go.Bar(x=srt.index.astype(str), y=srt.values, marker_color="#1f77b4", name="n_patrones"), row=1, col=2)
fig.update_xaxes(title_text="n_patrones", row=1, col=1)
fig.update_yaxes(title_text="cantidad de celdas", row=1, col=1)
fig.update_xaxes(title_text="cluster", type="category", row=1, col=2)
fig.update_yaxes(title_text="n_patrones", row=1, col=2)
fig.update_layout(title=f"Distribución de n_patrones (k={K})", height=450, showlegend=False)
fig.show()

print(resumen.n_patrones.describe().round(1))

count      100.0
mean      5027.6
std       3184.9
min        189.0
25%       2634.8
50%       4565.5
75%       7305.5
max      12126.0
Name: n_patrones, dtype: float64


In [8]:
# ---- Refinamiento recursivo del K-Means ----
CANT_PATRON_MAX = 4000   # máx. de imágenes/patrones por cluster
DIST_KM_MAX = 2000        # máx. distancia (km) de un patrón a su centroide
R_TIERRA = 6371.0

# Guardar el resultado base del k-means (no se pisa si se re-ejecuta esta celda)
if "centroids_km" not in globals():
    centroids_km, labels_km = centroids.copy(), labels.copy()

def dist_km_a(Xi, c):
    return np.arccos(np.clip(Xi @ c, -1, 1)) * R_TIERRA

def split2(Xi, n_iter=10):
    """Parte una celda en 2. Los centroides resultantes son puntos de la celda."""
    c = Xi.sum(axis=0); c /= np.linalg.norm(c)
    s1 = (Xi @ c).argmin()                # punto más lejano al centro
    s2 = (Xi @ Xi[s1]).argmin()           # punto más lejano a s1
    C = Xi[[s1, s2]].copy()
    for _ in range(n_iter):
        lab = (Xi @ C.T).argmax(axis=1)
        for j in range(2):
            m = lab == j
            if m.any():
                v = Xi[m].sum(axis=0)
                C[j] = v / np.linalg.norm(v)
    # Forzar que cada centroide sea un dato de la celda (el más cercano a la media)
    lab = (Xi @ C.T).argmax(axis=1)
    for j in range(2):
        idx = np.where(lab == j)[0]
        C[j] = Xi[idx[(Xi[idx] @ C[j]).argmax()]]
    lab = (Xi @ C.T).argmax(axis=1)
    return C, lab

def refinar(X, centroids0, labels0):
    final_C, final_idx = [], []
    pila = [(np.where(labels0 == j)[0], centroids0[j]) for j in range(len(centroids0))]
    while pila:
        idx, c = pila.pop()
        if len(idx) == 0:
            continue
        d = dist_km_a(X[idx], c)
        if len(idx) <= CANT_PATRON_MAX and d.max() <= DIST_KM_MAX:
            final_C.append(c); final_idx.append(idx)
        else:
            C2, lab2 = split2(X[idx])
            for j in range(2):
                pila.append((idx[lab2 == j], C2[j]))
    new_labels = np.empty(len(X), dtype=np.int32)
    for j, idx in enumerate(final_idx):
        new_labels[idx] = j
    return np.array(final_C), new_labels

centroids, labels = refinar(X, centroids_km, labels_km)
K = len(centroids)

# Reconstruir lo que usan las celdas siguientes
coords["cluster"] = labels
c_lat, c_lon = to_latlon(centroids)
cent = pd.DataFrame({"cluster": range(K), "lat": c_lat, "lon": c_lon,
                     "n": np.bincount(labels, minlength=K)})
ang = np.degrees(np.arccos(np.clip((X * centroids[labels]).sum(axis=1), -1, 1)))
dist_final = np.radians(ang) * R_TIERRA

print(f"clusters: {len(centroids_km)} (k-means) -> {K} (refinado)")
print(f"max patrones por cluster: {cent.n.max()} (límite {CANT_PATRON_MAX})")
print(f"max dist al centroide: {dist_final.max():.1f} km (límite {DIST_KM_MAX})")
print(f"distancia media al centroide: {dist_final.mean():.1f} km")
#cent


clusters: 100 (k-means) -> 213 (refinado)
max patrones por cluster: 3987 (límite 4000)
max dist al centroide: 1998.7 km (límite 2000)
distancia media al centroide: 192.5 km


In [9]:
def slerp_arc(a, b, n=24):
    omega = np.arccos(np.clip(a @ b, -1, 1))
    if omega < 1e-9:
        return np.array([a, b])
    t = np.linspace(0, 1, n)[:, None]
    return (np.sin((1 - t) * omega) * a + np.sin(t * omega) * b) / np.sin(omega)

def voronoi_edges(centroids):
    sv = SphericalVoronoi(centroids, radius=1.0)
    sv.sort_vertices_of_regions()
    seen, lats, lons = set(), [], []
    for region in sv.regions:
        for i in range(len(region)):
            u, v = region[i], region[(i + 1) % len(region)]
            key = (min(u, v), max(u, v))
            if key in seen:
                continue
            seen.add(key)
            la, lo = to_latlon(slerp_arc(sv.vertices[u], sv.vertices[v]))
            # cortar la línea si cruza el antimeridiano
            jump = np.where(np.abs(np.diff(lo)) > 180)[0]
            la, lo = list(la), list(lo)
            for j in reversed(jump):
                la.insert(j + 1, None); lo.insert(j + 1, None)
            lats += la + [None]; lons += lo + [None]
    return lats, lons

e_lat, e_lon = voronoi_edges(centroids)

fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=e_lat, lon=e_lon, mode="lines",
                            line=dict(width=1.5, color="#d62728"), name="Voronoi"))
fig.add_trace(go.Scattergeo(lat=cent.lat, lon=cent.lon, mode="markers+text",
                            text=cent.cluster.astype(str), textposition="top center",
                            marker=dict(size=9, color="black", symbol="x"),
                            customdata=cent.n, hovertemplate="cluster %{text}<br>n=%{customdata}<extra></extra>",
                            name="centroides"))
base_layout(fig, f"Centroides esféricos (k={K}) y diagrama de Voronoi").show()